# Valuing Actions by Estimating Probabilities (VAEP)

## Import Libraries

In [1]:
# Ensure that no .pyc files are generated
import sys

sys.dont_write_bytecode = True

In [ ]:
import warnings

import pandas as pd
import socceraction.vaep.features as fs
import socceraction.vaep.labels as lab
from socceraction import spadl
from tqdm import tqdm

from config import paths

In [9]:
warnings.filterwarnings(
    "ignore",
    message="DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.",
    category=FutureWarning,
)

## Load SPADL Data

In [5]:
SPADL_H5 = paths.SOCCERACTION_DIR / "spadl.h5"
FEATURES_H5 = paths.SOCCERACTION_DIR / "features.h5"
LABELS_H5 = paths.SOCCERACTION_DIR / "labels.h5"

In [6]:
all_games_df = pd.read_hdf(SPADL_H5, "games")

## Compute Features

In [10]:
x_fns = [
    fs.actiontype,
    fs.actiontype_onehot,
    fs.bodypart,
    fs.bodypart_onehot,
    fs.result,
    fs.result_onehot,
    fs.goalscore,
    fs.startlocation,
    fs.endlocation,
    fs.movement,
    fs.space_delta,
    fs.startpolar,
    fs.endpolar,
    fs.team,
    fs.time,
    fs.time_delta,
]

with pd.HDFStore(SPADL_H5) as spadl_store, pd.HDFStore(FEATURES_H5) as feature_store:
    for game in tqdm(list(all_games_df.itertuples()), desc=f"Generating and storing features in {FEATURES_H5}"):
        game_actions = spadl_store[f"actions/game_{game.game_id}"]
        gamestates = fs.gamestates(spadl.add_names(game_actions), 3)  # type: ignore
        gamestates = fs.play_left_to_right(gamestates, game.home_team_id)  # type: ignore

        X = pd.concat([fn(gamestates) for fn in x_fns], axis=1)
        feature_store.put(f"game_{game.game_id}", X, format="table")

Generating and storing features in C:\Users\cristian\Desktop\uchile\soccer-kpis\data\socceraction\features.h5:   0%|          | 0/2085 [00:00<?, ?it/s]

Generating and storing features in C:\Users\cristian\Desktop\uchile\soccer-kpis\data\socceraction\features.h5: 100%|██████████| 2085/2085 [14:54<00:00,  2.33it/s]


## Compute Labels

In [11]:
y_fns = [
    lab.scores,
    lab.concedes,
    lab.goal_from_shot,
]

with pd.HDFStore(SPADL_H5) as spadl_store, pd.HDFStore(LABELS_H5) as label_store:
    for game in tqdm(list(all_games_df.itertuples()), desc=f"Computing and storing labels in {LABELS_H5}"):
        game_actions = spadl_store[f"actions/game_{game.game_id}"]

        Y = pd.concat([fn(spadl.add_names(game_actions)) for fn in y_fns], axis=1)  # type: ignore
        label_store.put(f"game_{game.game_id}", Y, format="table")

Computing and storing labels in C:\Users\cristian\Desktop\uchile\soccer-kpis\data\socceraction\labels.h5: 100%|██████████| 2085/2085 [03:32<00:00,  9.80it/s]
